# Part 3 — μP (Maximal Update Parameterization): Tasks 3.1–3.3

**CS-GY 6923 · NYU Tandon · Spring 2026**

This notebook covers:
- **Task 3.1** — Conceptual overview of μP
- **Task 3.2** — Implementation: what changed vs. Standard Parameterization (SP)
- **Task 3.3** — μP LR sweep on the Tiny model + comparison with SP sweep

Key promise of μP: **the optimal learning rate found on the Tiny model transfers to all larger widths without retuning.**

In [ ]:
import os, sys, json, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Navigate to project root
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), 'ML-Final-Project'))
if not os.path.exists('configs'):
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
# Verify mup is installed
from mup import MuReadout, set_base_shapes, MuAdamW
print('mup package: OK')

from scripts.model    import ModelConfig
from scripts.mup_model import MupGPT, build_mup_model
print('MupGPT imports: OK')

### Change 1 — Attention Scale: `1/head_dim` instead of `1/√head_dim`

```python
# SP  (model.py)
scale = 1.0 / math.sqrt(self.head_dim)   # shrinks as width grows

# μP  (mup_model.py)
self._mup_scale = 1.0 / self.head_dim    # linear in head_dim
y = F.scaled_dot_product_attention(q, k, v, scale=self._mup_scale, ...)
```

### Change 2 — Output Layer: `MuReadout` instead of `nn.Linear`

```python
# SP  (model.py)
self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
self.transformer.wte.weight = self.lm_head.weight  # weight tying

# μP  (mup_model.py)
self.lm_head = MuReadout(config.d_model, config.vocab_size, bias=False)
# No weight tying — MuReadout gets a different LR multiplier than wte
```

### Change 3 — `build_mup_model()` registers base shapes

```python
def build_mup_model(config: ModelConfig) -> MupGPT:
    base_cfg  = ModelConfig(d_model=64,  n_layers=config.n_layers, ...)  # same depth, 64-wide
    delta_cfg = ModelConfig(d_model=128, n_layers=config.n_layers, ...)  # same depth, 128-wide
    base_model  = MupGPT(base_cfg)
    delta_model = MupGPT(delta_cfg)
    model       = MupGPT(config)
    set_base_shapes(model, base_model, delta=delta_model)   # registers μP infshapes
    return model
```

### Change 4 — `MuAdamW` instead of `AdamW`

```python
# SP
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, ...)

# μP
optimizer = MuAdamW(model.parameters(), lr=lr, ...)  # reads infshapes → applies lr multipliers
```

In [ ]:
# Show parameter counts for all 5 μP model sizes
configs = [
    ('tiny',   'configs/tiny.json'),
    ('small',  'configs/small.json'),
    ('medium', 'configs/medium.json'),
    ('large',  'configs/large.json'),
    ('xl',     'configs/xl.json'),
]

print(f"{'Name':<8} {'d_model':>8} {'n_layers':>9} {'n_heads':>8} {'SP params':>12} {'μP params':>12} {'Note'}")
print('-' * 75)

from scripts.model import GPT
for name, path in configs:
    with open(path) as f:
        cfg_dict = json.load(f)
    cfg = ModelConfig.from_dict(cfg_dict)
    sp_params  = GPT(cfg).count_parameters()
    mup_params = build_mup_model(cfg).count_parameters()
    delta = mup_params - sp_params
    print(f"{name:<8} {cfg.d_model:>8} {cfg.n_layers:>9} {cfg.n_heads:>8} "
          f"{sp_params:>12,} {mup_params:>12,}  +{delta:,} (no weight tying)")

In [ ]:
# Load both sweep results
with open('outputs/results/lr_sweep_results.json')     as f: sp_raw  = json.load(f)
with open('outputs/results/mup_lr_sweep_results.json') as f: mup_raw = json.load(f)

sp_results  = {float(k): v for k, v in sp_raw.items()}
mup_results = {float(k): v for k, v in mup_raw.items()}

lrs = sorted(sp_results.keys())

sp_best_lr  = min(sp_results,  key=sp_results.get)
mup_best_lr = min(mup_results, key=mup_results.get)

print('SP  LR Sweep Results (Task 2.2):')
print(f"  {'LR':>10}  {'Val Loss':>10}")
print('  ' + '-'*24)
for lr in lrs:
    marker = '  ← BEST' if lr == sp_best_lr else ''
    print(f"  {lr:>10.2e}  {sp_results[lr]:>10.4f}{marker}")

print()
print('μP  LR Sweep Results (Task 3.3):')
print(f"  {'LR':>10}  {'Val Loss':>10}")
print('  ' + '-'*24)
for lr in lrs:
    marker = '  ← BEST' if lr == mup_best_lr else ''
    print(f"  {lr:>10.2e}  {mup_results[lr]:>10.4f}{marker}")

lr_ratio = mup_best_lr / sp_best_lr
print(f'\nμP best LR / SP best LR = {mup_best_lr:.1e} / {sp_best_lr:.1e} = {lr_ratio:.1f}×')

### Side-by-Side Plot: SP vs μP LR Sweep

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, data, best, label, color, marker in [
    (axes[0], sp_results,  sp_best_lr,  'SP — Standard',        'steelblue',  'o'),
    (axes[1], mup_results, mup_best_lr, 'μP — MuP (Task 3.3)',  'darkorchid', 's'),
]:
    x = sorted(data.keys())
    y = [data[lr] for lr in x]
    ax.semilogx(x, y, f'{marker}-', color=color, linewidth=2, markersize=8, label=label)
    ax.axvline(best, color='tomato', linestyle='--', linewidth=1.8,
               label=f'Best LR = {best:.1e}  (val={data[best]:.4f})')
    for lr, v in zip(x, y):
        ax.annotate(f'{v:.3f}', (lr, v), textcoords='offset points',
                    xytext=(0, 9), ha='center', fontsize=8)
    ax.set_xlabel('Learning Rate (log scale)', fontsize=11)
    ax.set_ylabel('Validation Loss', fontsize=11)
    ax.set_title(f'{label}\nTiny Model — 20% epoch (310 steps)', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, which='both', alpha=0.3)

fig.suptitle('Task 3.3 — LR Sweep Comparison: SP vs μP', fontsize=13, fontweight='bold')
fig.tight_layout()
os.makedirs('outputs/plots', exist_ok=True)
fig.savefig('outputs/plots/lr_sweep_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/plots/lr_sweep_comparison.png')

### Overlay: Both Sweeps on the Same Axes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for data, best, label, color, marker in [
    (sp_results,  sp_best_lr,  'SP (Standard Param.)',       'steelblue',  'o'),
    (mup_results, mup_best_lr, 'μP (Maximal Update Param.)', 'darkorchid', 's'),
]:
    x = sorted(data.keys())
    y = [data[lr] for lr in x]
    ax.semilogx(x, y, f'{marker}-', color=color, linewidth=2, markersize=8, label=label)
    ax.axvline(best, color=color, linestyle=':', linewidth=1.5, alpha=0.7)
    ax.annotate(f'Best\n{best:.1e}', (best, ax.get_ylim()[1] if ax.get_ylim()[1] < 10 else 6.0),
                textcoords='offset points', xytext=(5, -20), fontsize=9, color=color)

ax.set_xlabel('Learning Rate (log scale)', fontsize=12)
ax.set_ylabel('Validation Loss (20% epoch)', fontsize=12)
ax.set_title('SP vs μP — LR Sensitivity on Tiny Model\n'
             'μP curve continues improving at higher LR; SP diverges at 1e-2', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
fig.savefig('outputs/plots/lr_sweep_overlay.png', dpi=150)
plt.show()
print('Saved: outputs/plots/lr_sweep_overlay.png')

In [ ]:
# Summary bar chart: best val loss comparison at the optimal LR for each method
labels = [f'{lr:.0e}' for lr in lrs]
sp_vals  = [sp_results[lr]  for lr in lrs]
mup_vals = [mup_results[lr] for lr in lrs]

x = np.arange(len(lrs))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_sp  = ax.bar(x - w/2, sp_vals,  w, label='SP',  color='steelblue',  alpha=0.8)
bars_mup = ax.bar(x + w/2, mup_vals, w, label='μP',  color='darkorchid', alpha=0.8)

# Highlight best bars
sp_best_idx  = sp_vals.index(min(sp_vals))
mup_best_idx = mup_vals.index(min(mup_vals))
bars_sp[sp_best_idx].set_edgecolor('gold');   bars_sp[sp_best_idx].set_linewidth(2.5)
bars_mup[mup_best_idx].set_edgecolor('gold'); bars_mup[mup_best_idx].set_linewidth(2.5)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Learning Rate', fontsize=12)
ax.set_ylabel('Validation Loss (20% epoch)', fontsize=12)
ax.set_title('SP vs μP Validation Loss at Each LR — Tiny Model\n'
             'Gold border = best per method', fontsize=11)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('outputs/plots/lr_sweep_bar_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/plots/lr_sweep_bar_comparison.png')

In [ ]:
from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, path, title in [
    (axes[0], 'outputs/plots/lr_sweep.png',     'SP LR Sweep (Task 2.2)'),
    (axes[1], 'outputs/plots/mup_lr_sweep.png', 'μP LR Sweep (Task 3.3)'),
]:
    if os.path.exists(path):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
    ax.axis('off')
fig.tight_layout()
plt.show()